In [8]:
!pip install transformers torch gradio sentencepiece accelerate pypdf

In [9]:
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM
from pypdf import PdfReader
import torch

# Modelo Qwen
MODEL_NAME = "Qwen/Qwen2-0.5B"

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Modelo
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Función para analizar sentimiento
def analizar_sentimiento(archivo):

    texto = ""

    # Leer PDF
    if archivo.name.endswith(".pdf"):

        reader = PdfReader(archivo.name)

        for page in reader.pages:

            contenido = page.extract_text()

            if contenido:
                texto += contenido

    # Leer TXT
    else:

        with open(archivo.name, "r", encoding="utf-8", errors="ignore") as f:
            texto = f.read()

    # Tomar solo una parte pequeña
    texto = texto[:500]

    # Prompt estilo zero-shot
    prompt = f"""
Classify the sentiment of this SEC filing as ONLY one word:

Positive
Negative

SEC filing:
{texto}

Sentiment:
"""

    # Tokenizar
    inputs = tokenizer(prompt, return_tensors="pt").input_ids

    # Generar respuesta
    output = model.generate(
        inputs,
        max_new_tokens=1,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decodificar solo la respuesta generada
    generated_text = tokenizer.decode(
        output[0][len(inputs[0]):],
        skip_special_tokens=True
    )

    return generated_text.strip()

# Crear app
app = gr.Interface(
    fn=analizar_sentimiento,
    inputs=gr.File(label="Upload SEC File"),
    outputs="text",
    title="📊 SEC Sentiment Analyzer",
    description="Sentiment analysis using Qwen2-0.5B"
)

# Crear liga pública
app.launch(share=True)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2f271958befadfac79.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
